# Build a bronze to silver to gold PySpark pipeline with joins, deduplication, and a gold materialized view

The growth team's daily-active-user dashboard has been off by 15% for a month and the PM has stopped trusting the number. You inherit the bronze clickstream and the silver layer it feeds. Three data-quality defects are baked in upstream and the team has no leverage to fix them at the SDK. You have one session to clean the bronze on read, broadcast-join the country lookup, deduplicate with the right tiebreaker, and ship a gold view that the dashboard can point at by tomorrow.

The hands-on work:

- Seed a managed Delta bronze table `workspace.default.labrun_medallion.bronze_clickstream` with 200 rows that include three deliberate defects: null user_id, mixed event_time types (epoch milliseconds and ISO-8601 strings), and ~12% duplicate (user_id, event_id) pairs.
- Stand up a 50-row enrichment table `workspace.default.labrun_medallion.user_country_map` for the broadcast join.
- Read bronze with PySpark, drop null user_id, normalize event_time to TIMESTAMP, broadcast-join enrichment, deduplicate on (user_id, event_id) keeping the latest event_time, and write to `silver_clickstream`.
- Capture the silver query plan in `silver_explain_text` so the validator can confirm a `BroadcastHashJoin` node is present.
- Create a `gold_dau_by_country` materialized view that aggregates daily active users by country.

**Time.** Plan for about 60 minutes hands-on. The slow part is reading the bronze rows and choosing the right cast pattern for `event_time`. Budget up to 100 minutes for the session window.

**Cost.** Zero dollars. PySpark on serverless and a materialized view refresh against the Starter warehouse both fit inside the Free Edition daily compute quota.

This lab maps to Databricks DE Associate Domain 3: Data Transformation and Modeling (~31% of exam weight, provisional).

**Runtime note.** The transformations need the in-notebook `spark` session, so this notebook must run inside a Databricks workspace notebook attached to serverless all-purpose compute. The setup cell guards against running outside Databricks.

In [ ]:
# NBVAL_SKIP
# Install labrun-checks and the Databricks SDK. Pinned versions per
# LAB-CREATION-BLUEPRINT.md build rules. Never use unpinned installs.

!pip install --quiet labrun-checks==0.6.0 databricks-sdk==0.40.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. UC identifiers use snake_case under the
# labrun_ prefix because Unity Catalog identifiers prefer snake_case.
# The silver query plan string is captured in silver_explain_text so the
# checkpoint can confirm the BroadcastHashJoin node is present.

import atexit
import getpass
import json
import sys
import time

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupResource,
)

LAB_ID = "pyspark-bronze-silver-gold-pipeline"
LAB_TAG_KEY = "labrun_lab_slug"
LAB_TAG_VALUE = LAB_ID

CATALOG = "workspace"
PARENT_SCHEMA = "default"
LAB_SCHEMA = "labrun_medallion"
LAB_SCHEMA_FQN = f"{CATALOG}.{PARENT_SCHEMA}.{LAB_SCHEMA}"
BRONZE_TABLE = "bronze_clickstream"
ENRICHMENT_TABLE = "user_country_map"
SILVER_TABLE = "silver_clickstream"
GOLD_VIEW = "gold_dau_by_country"
BRONZE_FQN = f"{LAB_SCHEMA_FQN}.{BRONZE_TABLE}"
ENRICHMENT_FQN = f"{LAB_SCHEMA_FQN}.{ENRICHMENT_TABLE}"
SILVER_FQN = f"{LAB_SCHEMA_FQN}.{SILVER_TABLE}"
GOLD_FQN = f"{LAB_SCHEMA_FQN}.{GOLD_VIEW}"

NUM_BRONZE_ROWS = 200
NUM_DISTINCT_USERS = 50
COUNTRIES = ["US", "CA", "GB", "DE", "FR", "JP", "AU", "BR", "IN", "MX"]

# Captured during Task 2 for Checkpoint 3 (BroadcastHashJoin assertion).
silver_explain_text = None

In [ ]:
# NBVAL_SKIP
# Register the session, validate Databricks credentials via the SDK, resolve
# the Starter SQL warehouse, and confirm the in-notebook spark session.
# Mirrors Lab 1 setup so the credential-validation muscle memory carries
# forward.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
databricks_host = getpass.getpass("Databricks workspace URL: ").strip()
databricks_token = getpass.getpass("Databricks personal access token (PAT): ").strip()

if not databricks_host or not databricks_token:
    print("Workspace URL and PAT are both required. Re-run this cell.")
    raise SystemExit(1)
if not databricks_host.startswith("https://"):
    databricks_host = f"https://{databricks_host}"

w = WorkspaceClient(host=databricks_host, token=databricks_token)

try:
    me = w.current_user.me()
except Exception as exc:
    print("Databricks credentials are missing or expired. CurrentUser.me() failed:")
    print(f"  {exc}")
    raise SystemExit(1)

CALLER_USER = me.user_name or (me.emails[0].value if me.emails else "unknown")
print(f"Credentials validated. Workspace user: {CALLER_USER}")

warehouses = list(w.warehouses.list())
if not warehouses:
    print("No SQL warehouses found. Recreate the Starter warehouse and re-run.")
    raise SystemExit(1)
WAREHOUSE = warehouses[0]
WAREHOUSE_ID = WAREHOUSE.id
print(f"SQL warehouse resolved: {WAREHOUSE.name} ({WAREHOUSE_ID})")
print("Materialized view refresh will route here; cold-start can take 30 to 90 seconds.")

try:
    spark  # type: ignore[name-defined]
except NameError:
    print("BLOCKED: this notebook must run inside a Databricks workspace.")
    print("The PySpark transformations need the in-notebook spark session.")
    print("Import this .ipynb into Free Edition, attach to serverless all-purpose")
    print("compute, and re-run.")
    raise SystemExit(1)
print("Spark session detected.")

DATABRICKS_CREDENTIALS = {
    "host": databricks_host,
    "token": databricks_token,
    "warehouse_id": WAREHOUSE_ID,
}


def run_sql(statement, warehouse_id=None, wait_seconds=180):
    wid = warehouse_id or WAREHOUSE_ID
    resp = w.statement_execution.execute_statement(
        warehouse_id=wid, statement=statement, wait_timeout="50s",
    )
    statement_id = resp.statement_id
    deadline = time.time() + wait_seconds
    while True:
        state = resp.status.state if resp.status else None
        if state == StatementState.SUCCEEDED:
            break
        if state in (StatementState.FAILED, StatementState.CANCELED, StatementState.CLOSED):
            err = resp.status.error.message if (resp.status and resp.status.error) else "no error message"
            raise RuntimeError(f"SQL failed ({state}): {err}\n  Statement: {statement}")
        if time.time() > deadline:
            raise TimeoutError(
                f"SQL did not finish in {wait_seconds}s. Last state: {state}."
            )
        time.sleep(2)
        resp = w.statement_execution.get_statement(statement_id)
    columns = []
    if resp.manifest and resp.manifest.schema and resp.manifest.schema.columns:
        columns = [c.name for c in resp.manifest.schema.columns]
    rows = []
    if resp.result and resp.result.data_array:
        rows = list(resp.result.data_array)
    return {"columns": columns, "rows": rows}


register(session_token=session_token, lab_id=LAB_ID, credentials=DATABRICKS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan. Order matters: the
# gold materialized view depends on silver, so the MV drops first; silver
# can be dropped before bronze/enrichment because the silver table holds
# its own copy (no FK in Delta). The schema drops last with CASCADE.


CLEANUP_MANIFEST = [
    CleanupResource(
        type="uc_materialized_view",
        id=GOLD_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP MATERIALIZED VIEW IF EXISTS {GOLD_FQN}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=SILVER_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {SILVER_FQN}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=ENRICHMENT_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {ENRICHMENT_FQN}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=BRONZE_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {BRONZE_FQN}\"",
    ),
    CleanupResource(
        type="uc_schema",
        id=LAB_SCHEMA_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP SCHEMA IF EXISTS {LAB_SCHEMA_FQN} CASCADE\"",
    ),
]


class DatabricksCleanupAdapter:
    def delete_resource(self, credentials, resource):
        rtype, rid = resource.type, resource.id
        if rtype == "uc_managed_table":
            run_sql(f"DROP TABLE IF EXISTS {rid}")
        elif rtype == "uc_materialized_view":
            run_sql(f"DROP MATERIALIZED VIEW IF EXISTS {rid}")
        elif rtype == "uc_volume":
            run_sql(f"DROP VOLUME IF EXISTS {rid}")
        elif rtype == "uc_volume_contents":
            volume_path = "/Volumes/" + rid.replace(".", "/") + "/"
            try:
                listing = list(w.files.list_directory_contents(volume_path))
            except Exception:
                return
            for entry in listing:
                try:
                    if entry.is_directory:
                        w.files.delete_directory(entry.path)
                    else:
                        w.files.delete(entry.path)
                except Exception:
                    pass
        elif rtype == "uc_schema":
            run_sql(f"DROP SCHEMA IF EXISTS {rid} CASCADE")
        else:
            raise ValueError(f"Unknown resource type {rtype!r}")

    def describe_resource(self, credentials, resource):
        rtype, rid = resource.type, resource.id
        if rtype == "uc_managed_table":
            catalog, schema, table = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.tables "
                f"WHERE table_catalog = '{catalog}' AND table_schema = '{schema}' "
                f"AND table_name = '{table}'"
            )
            return len(run_sql(sql)["rows"]) > 0
        if rtype == "uc_materialized_view":
            catalog, schema, view = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.views "
                f"WHERE table_catalog = '{catalog}' AND table_schema = '{schema}' "
                f"AND table_name = '{view}'"
            )
            return len(run_sql(sql)["rows"]) > 0
        if rtype == "uc_schema":
            catalog, schema = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.schemata "
                f"WHERE catalog_name = '{catalog}' AND schema_name = '{schema}'"
            )
            return len(run_sql(sql)["rows"]) > 0
        return False

    def tag_scan(self, credentials, lab_slug, region):
        arns = []
        queries = [
            (
                "SELECT catalog_name, schema_name FROM system.information_schema.schema_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}",
            ),
            (
                "SELECT catalog_name, schema_name, table_name FROM system.information_schema.table_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}.{row[2]}",
            ),
        ]
        for sql, fmt in queries:
            try:
                result = run_sql(sql)
            except Exception:
                continue
            for row in result["rows"]:
                arns.append(fmt(row))
        return arns


CLEANUP_ADAPTER = DatabricksCleanupAdapter()


def _atexit_cleanup():
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


orphans = CLEANUP_ADAPTER.tag_scan(DATABRICKS_CREDENTIALS, LAB_TAG_VALUE, "databricks")
if orphans:
    print(f"BLOCKED: existing UC objects tagged {LAB_TAG_KEY}={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("Run the cleanup cell at the bottom of this notebook first.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Seed bronze with three defects and stand up the enrichment table

The defects are deliberate. Bronze ingests data as-is from the upstream SDK; the medallion contract says silver is where you clean. Your job is to make sure bronze has the defects so silver has something to fix.

Build it in this order:

1. `CREATE SCHEMA workspace.default.labrun_medallion` and tag it.
2. `CREATE TABLE bronze_clickstream (user_id STRING, event_id STRING, event_time STRING, page STRING) USING DELTA`. Note `event_time` is `STRING` on purpose because bronze stores mixed types as text. Tag the table.
3. `CREATE TABLE user_country_map (user_id STRING, user_country STRING) USING DELTA`. Tag it.
4. Use the `seed_bronze()` and `seed_enrichment()` helpers below to populate both tables. The helpers produce deterministic data so the checkpoints can assert on counts and patterns.

The bronze defects you should expect after seeding:

- About 8 to 12 rows have `user_id = NULL`.
- About half the rows have `event_time` as an ISO-8601 string (e.g., `2026-05-13 10:15:22`); the other half have it as a numeric epoch-millis string (e.g., `1747130122000`).
- About 12% of the (user_id, event_id) pairs are duplicated with different `event_time` values.

In [ ]:
# NBVAL_SKIP
# Task 1: Schema, bronze table, enrichment table, seed data with defects.

import random

random.seed(42)

def seed_bronze():
    rows = []
    user_ids = [f"U-{i:03d}" for i in range(1, NUM_DISTINCT_USERS + 1)]
    base_epoch_ms = 1747100000000  # arbitrary anchor (May 2026)
    for i in range(NUM_BRONZE_ROWS):
        # ~5% NULL user_id (the duplicates and NULLs are deliberate defects).
        if random.random() < 0.05:
            uid = None
        else:
            uid = random.choice(user_ids)
        event_id = f"E-{i:05d}"
        # Roughly half epoch-millis, half ISO-8601, both stored as STRING.
        offset_ms = random.randint(0, 86_400_000)
        epoch_ms = base_epoch_ms + offset_ms
        if random.random() < 0.5:
            event_time = str(epoch_ms)
        else:
            secs = epoch_ms // 1000
            iso = time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime(secs))
            event_time = iso
        page = random.choice(["/home", "/search", "/cart", "/checkout", "/account"])
        rows.append((uid, event_id, event_time, page))
    # Inject duplicates: pick ~12% of rows that have non-null user_id,
    # duplicate them with a later event_time so the dedup tiebreaker matters.
    non_null = [r for r in rows if r[0] is not None]
    dup_count = int(len(rows) * 0.12)
    for src in random.sample(non_null, min(dup_count, len(non_null))):
        uid, event_id, event_time, page = src
        # Force the duplicate's event_time later so MAX(event_time) is the duplicate.
        try:
            base = int(event_time)
            later_time = str(base + 7_200_000)  # +2 hours
        except ValueError:
            secs = int(time.mktime(time.strptime(event_time, "%Y-%m-%d %H:%M:%S"))) + 7200
            later_time = time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime(secs))
        rows.append((uid, event_id, later_time, page))
    random.shuffle(rows)
    return rows[:NUM_BRONZE_ROWS]


def seed_enrichment():
    return [
        (f"U-{i:03d}", random.choice(COUNTRIES))
        for i in range(1, NUM_DISTINCT_USERS + 1)
    ]


# YOUR CODE: Run CREATE SCHEMA IF NOT EXISTS LAB_SCHEMA_FQN via run_sql().

# YOUR CODE: Run ALTER SCHEMA LAB_SCHEMA_FQN SET TAGS
# ('labrun_lab_slug' = LAB_TAG_VALUE) via run_sql().

# YOUR CODE: Run CREATE TABLE IF NOT EXISTS BRONZE_FQN with four columns
# (user_id STRING, event_id STRING, event_time STRING, page STRING)
# USING DELTA via run_sql(). event_time stays STRING on purpose.

# YOUR CODE: Tag the bronze table with ALTER TABLE BRONZE_FQN SET TAGS
# ('labrun_lab_slug' = LAB_TAG_VALUE).

# YOUR CODE: Run CREATE TABLE IF NOT EXISTS ENRICHMENT_FQN with two columns
# (user_id STRING, user_country STRING) USING DELTA via run_sql().

# YOUR CODE: Tag the enrichment table.

bronze_rows = seed_bronze()
enrichment_rows = seed_enrichment()

# YOUR CODE: Write bronze_rows into BRONZE_FQN. The simplest pattern is to
# create a Spark DataFrame from the list of tuples with column names
# ["user_id", "event_id", "event_time", "page"] and call
# .write.mode("append").saveAsTable(BRONZE_FQN). (Pre-create the table above
# so the schema matches.)

# YOUR CODE: Write enrichment_rows into ENRICHMENT_FQN similarly, with column
# names ["user_id", "user_country"].

bronze_count = run_sql(f"SELECT count(*) FROM {BRONZE_FQN}")["rows"][0][0]
enrichment_count = run_sql(f"SELECT count(*) FROM {ENRICHMENT_FQN}")["rows"][0][0]
print(f"Bronze rows: {int(bronze_count)}")
print(f"Enrichment rows: {int(enrichment_count)}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Schema tagged. Bronze table tagged with 200 rows containing
# the three deliberate defects (>=1 null user_id, mixed event_time types,
# >=10 duplicate (user_id, event_id) pairs). Enrichment table has 50 rows.


def checkpoint_1(session):
    try:
        schema_sql = (
            "SELECT 1 FROM system.information_schema.schemata "
            f"WHERE catalog_name = '{CATALOG}' AND schema_name = '{LAB_SCHEMA}'"
        )
        if not run_sql(schema_sql)["rows"]:
            return CheckpointResult(
                status="fail",
                error_reason=f"Schema {LAB_SCHEMA_FQN} not found.",
            )
        tag_sql = (
            "SELECT tag_value FROM system.information_schema.schema_tags "
            f"WHERE catalog_name = '{CATALOG}' AND schema_name = '{LAB_SCHEMA}' "
            f"AND tag_name = '{LAB_TAG_KEY}'"
        )
        if not any(r[0] == LAB_TAG_VALUE for r in run_sql(tag_sql)["rows"]):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Schema {LAB_SCHEMA_FQN} missing tag {LAB_TAG_KEY}={LAB_TAG_VALUE}. "
                    f"Apply ALTER SCHEMA SET TAGS."
                ),
            )

        for fqn, expected_count, descriptor in [
            (BRONZE_FQN, NUM_BRONZE_ROWS, "bronze"),
            (ENRICHMENT_FQN, NUM_DISTINCT_USERS, "enrichment"),
        ]:
            count_rows = run_sql(f"SELECT count(*) FROM {fqn}")["rows"]
            actual = int(count_rows[0][0]) if count_rows else 0
            if actual != expected_count:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"{fqn} has {actual} rows; expected {expected_count}. "
                        f"Drop and re-seed the {descriptor} table."
                    ),
                )

        null_uid_rows = run_sql(
            f"SELECT count(*) FROM {BRONZE_FQN} WHERE user_id IS NULL"
        )["rows"]
        null_uid_count = int(null_uid_rows[0][0]) if null_uid_rows else 0
        if null_uid_count < 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Bronze has no NULL user_id rows; expected at least 1 from the "
                    f"seed defect. Re-run seed_bronze() (the seed inserts ~5% nulls)."
                ),
            )

        dup_rows = run_sql(
            f"SELECT count(*) FROM (SELECT user_id, event_id FROM {BRONZE_FQN} "
            f"WHERE user_id IS NOT NULL GROUP BY user_id, event_id HAVING count(*) > 1)"
        )["rows"]
        dup_count = int(dup_rows[0][0]) if dup_rows else 0
        if dup_count < 10:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Bronze has only {dup_count} duplicate (user_id, event_id) pairs; "
                    f"expected at least 10 from the seed defect. Re-run seed_bronze()."
                ),
            )

        iso_rows = run_sql(
            f"SELECT count(*) FROM {BRONZE_FQN} WHERE event_time LIKE '____-__-__ %'"
        )["rows"]
        iso_count = int(iso_rows[0][0]) if iso_rows else 0
        epoch_rows = run_sql(
            f"SELECT count(*) FROM {BRONZE_FQN} "
            f"WHERE event_time NOT LIKE '____-__-__ %' AND event_time RLIKE '^[0-9]+$'"
        )["rows"]
        epoch_count = int(epoch_rows[0][0]) if epoch_rows else 0
        if iso_count == 0 or epoch_count == 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"event_time is not mixed-type: iso={iso_count}, epoch={epoch_count}. "
                    f"Both ISO-8601 strings and numeric epoch strings should be present."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint names which thing is missing: schema, schema tag, bronze count, bronze tags, enrichment count, or one of the three defects. The defects are baked into `seed_bronze()`; if they are absent, the seed function did not run or you wrote different rows.

</details>

<details><summary>Hint 2 (direction)</summary>

Five SQL statements (schema, schema tag, bronze table, bronze tag, enrichment table, enrichment tag) plus two writes via Spark. The Spark writes use `spark.createDataFrame(bronze_rows, ["user_id", "event_id", "event_time", "page"]).write.mode("append").saveAsTable(BRONZE_FQN)`. Same shape for enrichment.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1:

```python
run_sql(f"CREATE SCHEMA IF NOT EXISTS {LAB_SCHEMA_FQN}")
run_sql(f"ALTER SCHEMA {LAB_SCHEMA_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

run_sql(
    f"CREATE TABLE IF NOT EXISTS {BRONZE_FQN} ("
    f"  user_id STRING, event_id STRING, event_time STRING, page STRING"
    f") USING DELTA"
)
run_sql(f"ALTER TABLE {BRONZE_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

run_sql(
    f"CREATE TABLE IF NOT EXISTS {ENRICHMENT_FQN} ("
    f"  user_id STRING, user_country STRING"
    f") USING DELTA"
)
run_sql(f"ALTER TABLE {ENRICHMENT_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

(
    spark.createDataFrame(bronze_rows, ["user_id", "event_id", "event_time", "page"])
    .write.mode("append").saveAsTable(BRONZE_FQN)
)
(
    spark.createDataFrame(enrichment_rows, ["user_id", "user_country"])
    .write.mode("append").saveAsTable(ENRICHMENT_FQN)
)
```

</details>

**Wallet check.** Still at $0.00. 200 bronze rows and 50 enrichment rows. Free Edition has not noticed.

## Task 2: Clean, broadcast-join, deduplicate, write silver

This is the core medallion lesson. Read bronze, drop the null user_ids, normalize `event_time` to TIMESTAMP, broadcast-join with the enrichment table, deduplicate on (user_id, event_id) keeping the latest event_time, write to silver.

Three details that matter:

- `event_time` normalization needs to handle BOTH formats. Pattern: `coalesce(to_timestamp(col, "yyyy-MM-dd HH:mm:ss"), to_timestamp(from_unixtime(col.cast("long") / 1000)))`. The first cast returns NULL on numeric-epoch strings; the second cast returns NULL on ISO strings; coalesce picks whichever one worked.
- The broadcast hint is REQUIRED. Use `F.broadcast(enrichment_df)` on the join. Without it the Catalyst optimizer might still broadcast at this small size, but the checkpoint asserts the EXPLAIN plan literally contains `BroadcastHashJoin`.
- The dedup must keep the LATEST `event_time` per (user_id, event_id). The window-function pattern is what the exam tests: `Window.partitionBy(...).orderBy(F.desc("event_time"))` with `row_number() == 1`. A naive `.dropDuplicates(["user_id", "event_id"])` keeps an arbitrary row and the checkpoint will catch it.

After writing silver, capture the silver query plan into `silver_explain_text` so Checkpoint 3 can confirm the BroadcastHashJoin node is present.

In [ ]:
# NBVAL_SKIP
# Task 2: PySpark transformation chain. Build the silver DataFrame, write
# it to SILVER_FQN, then capture the .explain() output into
# silver_explain_text for Checkpoint 3.

from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_df = spark.read.table(BRONZE_FQN)
enrichment_df = spark.read.table(ENRICHMENT_FQN)

# YOUR CODE: Build cleaned_df by filtering bronze_df to drop rows where
# user_id IS NULL, and add a normalized event_ts TIMESTAMP column via
# coalesce(to_timestamp(F.col("event_time"), "yyyy-MM-dd HH:mm:ss"),
#          to_timestamp(F.from_unixtime(F.col("event_time").cast("long") / 1000))).

# YOUR CODE: Build joined_df by joining cleaned_df with F.broadcast(enrichment_df)
# on user_id (inner join is fine; only users present in enrichment matter for
# DAU-by-country).

# YOUR CODE: Build silver_df by applying a Window over (user_id, event_id)
# ordered by event_ts DESC, adding a row_number column, filtering rn == 1,
# and dropping the rn column. Select user_id, event_id, event_ts as
# event_time, page, user_country in that order.

# YOUR CODE: Write silver_df to SILVER_FQN via
# .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(SILVER_FQN).

# YOUR CODE: Tag the silver table with ALTER TABLE SILVER_FQN SET TAGS.

# Capture the EXPLAIN plan as a string for Checkpoint 3. The capture must
# happen on the silver_df, not after the write, so the BroadcastHashJoin is
# visible in the plan.
import io
import contextlib

global silver_explain_text
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    silver_df.explain(extended=False)
silver_explain_text = buf.getvalue()

silver_count = run_sql(f"SELECT count(*) FROM {SILVER_FQN}")["rows"][0][0]
print(f"Silver rows: {int(silver_count)}")
print(f"Silver plan captured: {len(silver_explain_text)} chars")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: silver has row count strictly less than 200; zero NULL user_id;
# every silver row's user_country matches the enrichment mapping; zero
# duplicate (user_id, event_id) pairs; event_time column is TIMESTAMP type.


def checkpoint_2(session):
    try:
        silver_sql = (
            "SELECT 1 FROM system.information_schema.tables "
            f"WHERE table_catalog = '{CATALOG}' AND table_schema = '{LAB_SCHEMA}' "
            f"AND table_name = '{SILVER_TABLE}'"
        )
        if not run_sql(silver_sql)["rows"]:
            return CheckpointResult(
                status="fail",
                error_reason=f"Silver table {SILVER_FQN} not found. Did saveAsTable run?",
            )

        count_rows = run_sql(f"SELECT count(*) FROM {SILVER_FQN}")["rows"]
        silver_count = int(count_rows[0][0]) if count_rows else 0
        if silver_count >= NUM_BRONZE_ROWS:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Silver has {silver_count} rows; expected fewer than "
                    f"{NUM_BRONZE_ROWS} (nulls and duplicates should have been dropped)."
                ),
            )
        if silver_count == 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Silver has 0 rows. The transformation dropped everything; "
                    f"check the inner-join target and the row_number filter."
                ),
            )

        null_rows = run_sql(
            f"SELECT count(*) FROM {SILVER_FQN} WHERE user_id IS NULL"
        )["rows"]
        null_count = int(null_rows[0][0]) if null_rows else 0
        if null_count > 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Silver has {null_count} rows with NULL user_id; expected 0. "
                    f"Filter bronze with .filter(F.col('user_id').isNotNull())."
                ),
            )

        dup_rows = run_sql(
            f"SELECT count(*) FROM (SELECT user_id, event_id FROM {SILVER_FQN} "
            f"GROUP BY user_id, event_id HAVING count(*) > 1)"
        )["rows"]
        dup_count = int(dup_rows[0][0]) if dup_rows else 0
        if dup_count > 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Silver has {dup_count} duplicate (user_id, event_id) pairs; "
                    f"expected 0. Apply the window-function dedup keeping rn = 1."
                ),
            )

        cols_sql = (
            "SELECT column_name, data_type FROM system.information_schema.columns "
            f"WHERE table_catalog = '{CATALOG}' AND table_schema = '{LAB_SCHEMA}' "
            f"AND table_name = '{SILVER_TABLE}' ORDER BY ordinal_position"
        )
        cols = [(r[0], (r[1] or "").upper()) for r in run_sql(cols_sql)["rows"]]
        col_map = dict(cols)
        if "TIMESTAMP" not in col_map.get("event_time", ""):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Silver event_time column type is {col_map.get('event_time')!r}; "
                    f"expected TIMESTAMP. Cast event_time via coalesce(to_timestamp(...), ...) "
                    f"before saveAsTable."
                ),
            )
        if "user_country" not in col_map:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Silver is missing user_country column. The broadcast join with "
                    f"the enrichment table should add it."
                ),
            )

        mismatch_rows = run_sql(
            f"SELECT count(*) FROM {SILVER_FQN} s LEFT JOIN {ENRICHMENT_FQN} e "
            f"ON s.user_id = e.user_id WHERE s.user_country != e.user_country "
            f"OR e.user_id IS NULL"
        )["rows"]
        mismatch = int(mismatch_rows[0][0]) if mismatch_rows else 0
        if mismatch > 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{mismatch} silver rows have user_country that does not match "
                    f"the enrichment mapping for that user_id. The join may have "
                    f"used a stale enrichment_df copy or assigned a constant."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint message tells you exactly which assertion failed: row count, null filter, dedup, column type, or country mismatch. Each is a different piece of the transformation chain.

</details>

<details><summary>Hint 2 (direction)</summary>

Three intermediate DataFrames. First, `cleaned_df = bronze_df.filter(F.col("user_id").isNotNull()).withColumn("event_ts", coalesce(...))`. Second, `joined_df = cleaned_df.join(F.broadcast(enrichment_df), "user_id")`. Third, `silver_df = joined_df.withColumn("rn", F.row_number().over(Window.partitionBy("user_id", "event_id").orderBy(F.desc("event_ts")))).filter("rn = 1").drop("rn").select("user_id", "event_id", F.col("event_ts").alias("event_time"), "page", "user_country")`. Then `.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(SILVER_FQN)`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2:

```python
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_df = spark.read.table(BRONZE_FQN)
enrichment_df = spark.read.table(ENRICHMENT_FQN)

cleaned_df = (
    bronze_df
    .filter(F.col("user_id").isNotNull())
    .withColumn(
        "event_ts",
        F.coalesce(
            F.to_timestamp(F.col("event_time"), "yyyy-MM-dd HH:mm:ss"),
            F.to_timestamp(F.from_unixtime(F.col("event_time").cast("long") / 1000)),
        ),
    )
)

joined_df = cleaned_df.join(F.broadcast(enrichment_df), "user_id")

window = Window.partitionBy("user_id", "event_id").orderBy(F.desc("event_ts"))
silver_df = (
    joined_df
    .withColumn("rn", F.row_number().over(window))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .select(
        "user_id",
        "event_id",
        F.col("event_ts").alias("event_time"),
        "page",
        "user_country",
    )
)

(
    silver_df
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_FQN)
)
run_sql(f"ALTER TABLE {SILVER_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

import io, contextlib
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    silver_df.explain(extended=False)
silver_explain_text = buf.getvalue()
```

</details>

**Wallet check.** Still at $0.00. The transformation read 200 rows, dropped some, joined a 50-row dim, deduplicated, and wrote a smaller silver table. Serverless ran for about a second.

## Task 3: Prove the broadcast hint landed and the dedup kept the latest event

This task is verification, not new code. The validator does two things:

1. Parses `silver_explain_text` (captured in Task 2) and asserts the substring `BroadcastHashJoin` appears. The exam tests this; the explicit `F.broadcast()` hint is the muscle-memory pattern.
2. For at least 5 duplicate (user_id, event_id) pairs from bronze, confirms the silver row's `event_time` equals `MAX(event_time)` from bronze for that key. This catches the `.dropDuplicates()` mistake (which keeps an arbitrary row, not the latest).

If you used `F.broadcast()` and the window-function dedup in Task 2, this checkpoint should pass without further code.

In [ ]:
# NBVAL_SKIP
# Task 3: No new code; the checkpoint inspects silver_explain_text and runs
# a per-key MAX comparison against bronze.

if silver_explain_text is None:
    print("WARNING: silver_explain_text is None. Re-run Task 2 to capture the plan.")
else:
    print(f"silver_explain_text length: {len(silver_explain_text)} chars")
    print()
    print("First 400 chars of the plan (search for BroadcastHashJoin):")
    print(silver_explain_text[:400])

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: silver_explain_text contains 'BroadcastHashJoin'; for each
# of at least 5 duplicate (user_id, event_id) pairs from bronze, the silver
# row's event_time equals the MAX(event_time) from bronze for that pair.


def checkpoint_3(session):
    try:
        if silver_explain_text is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "silver_explain_text is None. Capture it after building "
                    "silver_df: redirect stdout while calling "
                    "silver_df.explain(extended=False) and store the result."
                ),
            )
        if "BroadcastHashJoin" not in silver_explain_text:
            preview = silver_explain_text[:400].replace("\n", " ")
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"silver_explain_text does not contain 'BroadcastHashJoin'. "
                    f"The Catalyst optimizer used a different join strategy. "
                    f"Wrap the enrichment_df with F.broadcast(enrichment_df) on the "
                    f"join. Plan preview: {preview!r}"
                ),
            )

        # Compare per-key event_time. Convert bronze event_time strings to
        # TIMESTAMP using the same coalesce pattern so the MAX is comparable.
        compare_sql = f"""
            WITH bronze_max AS (
              SELECT
                user_id,
                event_id,
                MAX(
                  coalesce(
                    to_timestamp(event_time, 'yyyy-MM-dd HH:mm:ss'),
                    to_timestamp(from_unixtime(CAST(event_time AS BIGINT) / 1000))
                  )
                ) AS max_event_time
              FROM {BRONZE_FQN}
              WHERE user_id IS NOT NULL
              GROUP BY user_id, event_id
              HAVING count(*) > 1
            )
            SELECT count(*) AS mismatches
            FROM bronze_max b
            JOIN {SILVER_FQN} s
              ON s.user_id = b.user_id AND s.event_id = b.event_id
            WHERE s.event_time != b.max_event_time
        """
        mismatch_rows = run_sql(compare_sql)["rows"]
        mismatches = int(mismatch_rows[0][0]) if mismatch_rows else 0
        if mismatches > 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{mismatches} duplicated (user_id, event_id) pairs in silver have "
                    f"an event_time that is NOT the MAX from bronze. The dedup kept the "
                    f"wrong row. Use Window.orderBy(F.desc('event_ts')) with row_number == 1."
                ),
            )

        dup_keys_rows = run_sql(
            f"SELECT count(*) FROM (SELECT user_id, event_id FROM {BRONZE_FQN} "
            f"WHERE user_id IS NOT NULL GROUP BY user_id, event_id "
            f"HAVING count(*) > 1)"
        )["rows"]
        dup_keys = int(dup_keys_rows[0][0]) if dup_keys_rows else 0
        if dup_keys < 5:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Bronze has only {dup_keys} duplicate keys; checkpoint requires "
                    f"at least 5 to validate the tiebreaker. Re-run seed_bronze()."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint either complains about a missing `BroadcastHashJoin` in the plan (the explicit hint is absent) or about per-key event_time mismatches (the dedup kept the wrong row). Both are Task 2 issues, not Task 3 issues.

</details>

<details><summary>Hint 2 (direction)</summary>

If BroadcastHashJoin is missing, your join was `cleaned_df.join(enrichment_df, "user_id")` without `F.broadcast(...)`. If the per-key MAX comparison fails, you used `.dropDuplicates()` instead of the window-function pattern, or the Window's `orderBy` was ascending instead of descending.

</details>

<details><summary>Hint 3 (spoiler)</summary>

If you need to retry Task 2:

```python
joined_df = cleaned_df.join(F.broadcast(enrichment_df), "user_id")
window = Window.partitionBy("user_id", "event_id").orderBy(F.desc("event_ts"))
silver_df = (
    joined_df.withColumn("rn", F.row_number().over(window))
    .filter(F.col("rn") == 1).drop("rn")
    .select("user_id", "event_id", F.col("event_ts").alias("event_time"),
            "page", "user_country")
)
```

The two pieces that matter: `F.broadcast(enrichment_df)` and `F.desc("event_ts")` in the Window.

</details>

**Wallet check.** Still at $0.00. The validator runs two SELECTs against the warehouse and a substring search on `silver_explain_text` (a Python string in your kernel memory). Free Edition has not flinched.

## Task 4: Create the gold materialized view

The dashboard wants one row per (event_date, user_country) with the count of distinct daily active users. Create a Unity Catalog materialized view on the silver table:

```sql
CREATE OR REPLACE MATERIALIZED VIEW workspace.default.labrun_medallion.gold_dau_by_country AS
SELECT
  date(event_time) AS event_date,
  user_country,
  count(DISTINCT user_id) AS daily_active_users
FROM workspace.default.labrun_medallion.silver_clickstream
GROUP BY date(event_time), user_country;
```

Then tag the view:

```sql
ALTER MATERIALIZED VIEW workspace.default.labrun_medallion.gold_dau_by_country
SET TAGS ('labrun_lab_slug' = 'pyspark-bronze-silver-gold-pipeline');
```

The view refreshes on the SQL warehouse. If the warehouse is cold, the first `SELECT` after CREATE can take 30 to 90 seconds. The validator polls.

In [ ]:
# NBVAL_SKIP
# Task 4: Create the gold materialized view aggregating DAU by country.

# YOUR CODE: Run CREATE OR REPLACE MATERIALIZED VIEW GOLD_FQN AS SELECT
# date(event_time) AS event_date, user_country,
# count(DISTINCT user_id) AS daily_active_users FROM SILVER_FQN
# GROUP BY date(event_time), user_country via run_sql().

# YOUR CODE: Tag the view with ALTER MATERIALIZED VIEW GOLD_FQN SET TAGS
# ('labrun_lab_slug' = LAB_TAG_VALUE) via run_sql().

print(f"Materialized view created: {GOLD_FQN}")
print("Asking the Starter warehouse to put on its reading glasses...")

result = run_sql(f"SELECT count(*) FROM {GOLD_FQN}", wait_seconds=240)
print(f"gold_dau_by_country rows: {int(result['rows'][0][0])}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: information_schema.views shows gold_dau_by_country with
# is_materialized=true; the view has the three expected columns; the sum of
# daily_active_users equals count(DISTINCT user_id, event_date) on silver.


def checkpoint_4(session):
    try:
        view_sql = (
            "SELECT 1 FROM system.information_schema.views "
            f"WHERE table_catalog = '{CATALOG}' AND table_schema = '{LAB_SCHEMA}' "
            f"AND table_name = '{GOLD_VIEW}'"
        )
        if not run_sql(view_sql)["rows"]:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"View {GOLD_FQN} not found in information_schema.views. "
                    f"Did CREATE OR REPLACE MATERIALIZED VIEW run?"
                ),
            )

        cols_sql = (
            "SELECT column_name FROM system.information_schema.columns "
            f"WHERE table_catalog = '{CATALOG}' AND table_schema = '{LAB_SCHEMA}' "
            f"AND table_name = '{GOLD_VIEW}' ORDER BY ordinal_position"
        )
        col_names = [r[0] for r in run_sql(cols_sql)["rows"]]
        expected_cols = {"event_date", "user_country", "daily_active_users"}
        missing = expected_cols - set(col_names)
        if missing:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Gold view is missing columns: {sorted(missing)}. "
                    f"Got columns: {col_names}. Re-create with the three SELECT "
                    f"projections: event_date, user_country, daily_active_users."
                ),
            )

        gold_sum_rows = run_sql(
            f"SELECT sum(daily_active_users) FROM {GOLD_FQN}",
            wait_seconds=240,
        )["rows"]
        gold_sum = int(gold_sum_rows[0][0]) if gold_sum_rows and gold_sum_rows[0][0] is not None else 0

        silver_parity_rows = run_sql(
            f"SELECT count(*) FROM ("
            f"  SELECT DISTINCT user_id, date(event_time) "
            f"  FROM {SILVER_FQN}"
            f")"
        )["rows"]
        silver_parity = int(silver_parity_rows[0][0]) if silver_parity_rows else 0

        if gold_sum != silver_parity:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"sum(daily_active_users) in gold is {gold_sum}; expected "
                    f"{silver_parity} (count of DISTINCT user_id, event_date in silver). "
                    f"The aggregation may be missing the DISTINCT or grouping by the "
                    f"wrong columns."
                ),
            )
        return CheckpointResult(status="pass")
    except TimeoutError as exc:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Gold view query timed out: {exc}. The Starter warehouse may be "
                f"cold; re-run this cell."
            ),
        )
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint message names which thing failed. If the view is missing, the CREATE statement did not run. If the column set is wrong, the SELECT projection used different names. If the parity check fails, the GROUP BY or the DISTINCT is off.

</details>

<details><summary>Hint 2 (direction)</summary>

Two SQL statements. The CREATE OR REPLACE MATERIALIZED VIEW uses `date(event_time) AS event_date`, `user_country`, and `count(DISTINCT user_id) AS daily_active_users`. The ALTER MATERIALIZED VIEW SET TAGS applies the lab tag. Use `run_sql()` and accept the cold-start delay.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4:

```python
run_sql(
    f"CREATE OR REPLACE MATERIALIZED VIEW {GOLD_FQN} AS "
    f"SELECT date(event_time) AS event_date, user_country, "
    f"  count(DISTINCT user_id) AS daily_active_users "
    f"FROM {SILVER_FQN} "
    f"GROUP BY date(event_time), user_country"
)
run_sql(
    f"ALTER MATERIALIZED VIEW {GOLD_FQN} "
    f"SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')"
)
```

</details>

**Wallet check.** Still at $0.00. The MV refresh ran one aggregation query on the Starter warehouse against a silver table that fits in memory. A few warehouse-seconds. Daily quota has not noticed.

## Cleanup

Time to tear it all down. The cell below drops the materialized view first (silver can't be dropped while the MV depends on it), then silver, enrichment, bronze, and the schema. Re-running is safe.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation
# order using the inline Databricks adapter.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_destroyed = 0
standard_destroyed = len(CLEANUP_MANIFEST) - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Databricks workspace may still hold lab objects. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: $0.00.** Two managed Delta tables, one transformation chain, one materialized view, and a few seconds of warehouse compute. Free Edition swallowed the whole session. The expensive part was the wall-clock time you spent deciding which defects belonged at bronze vs silver vs gold; that is the medallion contract in microcosm and the exam tests it.

## Reflection

These are not graded. They are for you.

1. Walk through what Spark does when you write `df_a.join(F.broadcast(df_b), "key")` vs `df_a.join(df_b, "key")` when `df_b` is 50 rows. Name the join strategies the optimizer can choose (BroadcastHashJoin, SortMergeJoin, ShuffleHashJoin), the shuffle behavior of each, and when the explicit `broadcast()` hint matters even though the optimizer could have chosen the same plan.

2. Your team has a (user_id, event_id) duplicate problem upstream that the SDK team will not fix for at least a quarter. Sketch three places you could deduplicate (at bronze ingest, at silver transformation, at gold aggregation) and the trade-offs of each. Why does the medallion convention prefer silver?

## Exam-Style Practice

**Question 1 (Domain 3, deduplication strategy):**

A data engineer wants to deduplicate rows in a DataFrame by `(user_id, event_id)` keeping the row with the latest `event_time` per key. Which PySpark expression is correct?

A. `df.dropDuplicates(["user_id", "event_id"])`
B. `df.orderBy(F.desc("event_time")).dropDuplicates(["user_id", "event_id"])`
C. `df.withColumn("rn", F.row_number().over(Window.partitionBy("user_id", "event_id").orderBy(F.desc("event_time")))).filter("rn = 1").drop("rn")`
D. `df.groupBy("user_id", "event_id").agg(F.max("event_time").alias("event_time"))`

<details><summary>Show answer</summary>

**Correct: C.**

- A is wrong: `dropDuplicates` keeps an arbitrary representative row, not the latest. It works for "any one row per key" but not for "the latest row per key."
- B is wrong: `dropDuplicates` after `orderBy` does not preserve order across the shuffle that `dropDuplicates` itself performs. The result is still effectively non-deterministic.
- C is correct: the window function with `row_number() == 1` over a partition ordered by event_time DESC keeps the latest row per `(user_id, event_id)`. This is the canonical Spark dedup pattern when a tiebreaker is required.
- D collapses the row to one column (event_time) and loses every other column; you cannot recover the original row's `user_country` or other fields after the groupBy.

</details>

**Question 2 (Domain 3, broadcast join semantics):**

A team's silver pipeline joins a 200 GB fact DataFrame with a 5 MB dimension DataFrame on `user_id`. The pipeline is slow; the Spark UI shows a sort-merge join with heavy shuffle on the fact side. The team's engineer adds `F.broadcast(dim_df)` to the join. What changes?

A. Spark broadcasts the dim_df to every executor and performs a `BroadcastHashJoin`, avoiding the shuffle of the fact DataFrame entirely.
B. Spark broadcasts the fact_df because broadcast joins always send the larger side; the team is mis-applying the hint.
C. Spark falls back to a `SortMergeJoin` because the dim_df is too large to broadcast (default threshold is 10 MB).
D. The hint has no effect; Spark always chooses the join strategy automatically based on table statistics.

<details><summary>Show answer</summary>

**Correct: A.**

- A is correct: `F.broadcast(dim_df)` instructs Spark to broadcast the dim DataFrame to every executor and run a `BroadcastHashJoin`. The fact DataFrame is not shuffled; each executor probes its local hash table of the broadcast side. This is the canonical optimization for fact-times-dim joins where the dim is small enough to fit in memory.
- B is wrong: broadcast joins broadcast the side you explicitly hint, or the smaller side under autoBroadcastJoinThreshold. The hint sends the SMALLER side.
- C is wrong at 5 MB: the default `spark.sql.autoBroadcastJoinThreshold` is 10 MB and 5 MB fits under it. The hint forces broadcast regardless of threshold.
- D is wrong: the hint overrides the optimizer's default choice. The optimizer respects explicit `broadcast()` hints unless the broadcast side exceeds the broadcast hint threshold (a separate, usually larger value).

</details>